
# Mental Rotation Prompt Experiments (Qwen3.5-0.8B)

This notebook documents **phase-wise prompt optimizations** for the **Mental Rotation** benchmark on **Qwen3.5-0.8B**. Mental Rotation is a binary (A/B) image matching task requiring spatial reasoning.

- **83 trials**: 49 type-2 (2D silhouettes: rabbits, ducks) + 34 type-3 (3D block shapes).
- **Chance level:** 50% (binary choice).
- **Task structure:** Each trial shows a rotated reference image and two options — the correct match (same shape, 0° orientation) and its mirror image. The model must determine which option matches the rotated reference.
- **Key difficulty:** Distinguishing rotation from reflection (chirality). Higher rotation angles are harder.
- **Artifacts:** per-phase CSVs and logs under `results/mrot-phases/`.

## Phases

| Phase | Name | Description |
|-------|------|-------------|
| 0 | Baseline | Manifest prompts + Qwen default system prompt + thinking disabled |
| 1 | Structured prompt | Reference / question / options clearly separated |
| 2 | Enhanced parsing | Reverse-scan, "image X" patterns, last-letter fallback |
| 3 | Task-specific system prompt | "Spatial reasoning assistant" |
| 4 | Mirror awareness hint | Chirality cue: "one is rotated, the other is flipped" |
| 5 | Feature-based CoT | Identify features → check orientation → choose |


In [1]:
from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)

RESULTS = ROOT / "results" / "prompt_optimization" / "mental-rotation" / "qwen-0.8b"
# Load prompt builders from the experiment script
_spec = importlib.util.spec_from_file_location(
    "mrot_phases", ROOT / "scripts" / "prompt_optimization" / "mental-rotation" / "experiment_qwen.py"
)
mp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(mp)

manifest_rows = mp.load_manifest(ROOT / "data")
IMAGE_DIR = ROOT / "data" / "assets" / "2026-03-26" / "visual" / "mental-rotation"
TRIALS_LIST = mp.build_trials(manifest_rows, IMAGE_DIR)
TRIALS = {t["item_uid"]: t for t in TRIALS_LIST}


def build_prompt(phases: set[int], item_uid: str) -> str:
    """Reconstruct the exact prompt used for a given phase combination."""
    t = TRIALS[item_uid]
    row = t["row"]
    pp = t["prompt_phrase"]
    if 1 in phases:
        p = mp.build_prompt_phase1(row, pp)
    else:
        p = mp.build_prompt_baseline(row, pp)
    if 4 in phases:
        p = mp.apply_phase4_mirror_hint(p)
    if 5 in phases:
        p = mp.apply_phase5_cot(p, t["trial_type"])
    if 3 in phases:
        p = mp.apply_phase3_system(p)
    return p


def load_result(csv_name: str, item_uid: str) -> dict | None:
    path = RESULTS / csv_name
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["item_uid"] == item_uid:
                return row
    return None


## Summary: All Runs

Aggregate accuracy, parse rate, and delta vs baseline across all phase configurations.

In [2]:
PHASE_CSVS = [
    ("phase_baseline.csv", "0 — Baseline"),
    ("phase_1.csv", "1 — Structured prompt"),
    ("phase_2.csv", "2 — Enhanced parsing"),
    ("phase_3.csv", "3 — Spatial system prompt"),
    ("phase_4.csv", "4 — Mirror awareness hint"),
    ("phase_5.csv", "5 — Feature-based CoT"),
    ("phase_1_2_3.csv", "1+2+3 — Structured + parse + sys"),
    ("phase_1_4.csv", "1+4 — Structured + mirror hint"),
    ("phase_1_2_3_4_5.csv", "1+2+3+4+5 — All"),
]


def summarize_all():
    rows = []
    baseline_acc = None
    for csv_name, label in PHASE_CSVS:
        path = RESULTS / csv_name
        if not path.exists():
            continue
        with open(path, newline="", encoding="utf-8") as f:
            data = list(csv.DictReader(f))
        n = len(data)
        if n == 0:
            continue
        correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
        parsed = sum(1 for r in data if r["parsed"].lower() in ("true", "1"))
        acc = correct / n
        pr = parsed / n
        if baseline_acc is None:
            baseline_acc = acc
        delta = "—" if acc == baseline_acc and label.startswith("0") else f"{(acc - baseline_acc)*100:+.1f} pp"
        rows.append({"Phase": label, "N": n, "Accuracy": f"{acc:.1%}", "Parse %": f"{pr:.1%}", "Δ vs baseline": delta})
    return pd.DataFrame(rows)


df_summary = summarize_all()
df_summary.style.hide(axis="index")

Phase,N,Accuracy,Parse %,Δ vs baseline
0 — Baseline,83,59.0%,100.0%,—
1 — Structured prompt,83,60.2%,100.0%,+1.2 pp
2 — Enhanced parsing,83,59.0%,100.0%,+0.0 pp
3 — Spatial system prompt,83,59.0%,100.0%,+0.0 pp
4 — Mirror awareness hint,83,59.0%,100.0%,+0.0 pp
5 — Feature-based CoT,83,60.2%,100.0%,+1.2 pp
1+2+3 — Structured + parse + sys,83,59.0%,100.0%,+0.0 pp
1+4 — Structured + mirror hint,83,59.0%,100.0%,+0.0 pp
1+2+3+4+5 — All,83,59.0%,100.0%,+0.0 pp


## Per-Trial-Type Breakdown

Accuracy by trial type for each phase:
- **Type 2**: 2D silhouettes (rabbits, ducks) — simpler, distinctive asymmetric features
- **Type 3**: 3D block shapes — harder, more abstract

In [3]:
TYPE_LABELS = {"2": "2D silhouettes", "3": "3D shapes"}


def breakdown_by_trial_type(csv_name: str, label: str):
    path = RESULTS / csv_name
    if not path.exists():
        print(f"  {csv_name} not found — skipped")
        return
    with open(path, newline="", encoding="utf-8") as f:
        data = list(csv.DictReader(f))
    by_type: dict[str, dict] = {}
    for r in data:
        tt = r["trial_type"]
        rec = by_type.setdefault(tt, {"N": 0, "Correct": 0, "Parsed": 0})
        rec["N"] += 1
        rec["Correct"] += int(r["is_correct"].lower() in ("true", "1"))
        rec["Parsed"] += int(r["parsed"].lower() in ("true", "1"))
    rows = []
    for tt, rec in sorted(by_type.items(), key=lambda kv: kv[1]["Correct"] / max(kv[1]["N"], 1), reverse=True):
        n = rec["N"]
        rows.append({"Trial Type": TYPE_LABELS.get(tt, tt), "N": n, "Accuracy": f"{rec['Correct']/n:.0%}", "Parse %": f"{rec['Parsed']/n:.0%}"})
    total_n = sum(rec["N"] for rec in by_type.values())
    total_c = sum(rec["Correct"] for rec in by_type.values())
    total_p = sum(rec["Parsed"] for rec in by_type.values())
    rows.append({"Trial Type": "OVERALL", "N": total_n, "Accuracy": f"{total_c/total_n:.0%}", "Parse %": f"{total_p/total_n:.0%}"})
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))


for csv_name, label in PHASE_CSVS:
    breakdown_by_trial_type(csv_name, label)


  0 — Baseline
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      53%    100%
       OVERALL 83      59%    100%

  1 — Structured prompt
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      56%    100%
       OVERALL 83      60%    100%

  2 — Enhanced parsing
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      53%    100%
       OVERALL 83      59%    100%

  3 — Spatial system prompt
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      53%    100%
       OVERALL 83      59%    100%

  4 — Mirror awareness hint
    Trial Type  N Accuracy Parse %
2D silhouettes 49      63%    100%
     3D shapes 34      53%    100%
       OVERALL 83      59%    100%

  5 — Feature-based CoT
    Trial Type  N Accuracy Parse %
2D silhouettes 49      61%    100%
     3D shapes 34      59%    100%
       OVERALL 83      60%    100%

  1+2+3 — S

---

## Phase 0 — Baseline

Uses the manifest prompt with the reference image inlined:

```
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>
```

**Note:** `<image0>` is the reference (rotated shape), `<image1>` and `<image2>` are the two options at 0°.

In [4]:
# Phase 0: 2D silhouette example
uid0_2d = "mrot_2d_rabbit_080_Rp-080"
print("=== PROMPT (Phase 0, 2D rabbit 80°) ===")
print(build_prompt(set(), uid0_2d))
r = load_result("phase_baseline.csv", uid0_2d)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet — run the experiment first)")

=== PROMPT (Phase 0, 2D rabbit 80°) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


In [5]:
# Phase 0: 3D shape example
uid0_3d = "mrot_3d_shape_280_ap1-280"
print("=== PROMPT (Phase 0, 3D shape 280°) ===")
print(build_prompt(set(), uid0_3d))
r = load_result("phase_baseline.csv", uid0_3d)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 0, 3D shape 280°) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


---

## Phase 1 — Structured Prompt

Separates the reference image from options with an explicit question:

```
Reference image:
<image0>

Which option shows the same shape as the reference, just rotated?

A: <image1>
B: <image2>

Answer with one letter: A or B.
```

**Rationale:** The baseline prompt is a single line with mixed text and image tokens. Separating reference from options and using the word "rotated" explicitly frames the spatial task.

In [6]:
uid1 = "mrot_2d_duck_200_Dp-200"
print("=== PROMPT (Phase 1, 2D duck 200°) ===")
print(build_prompt({1}, uid1))
r = load_result("phase_1.csv", uid1)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1, 2D duck 200°) ===
Reference image:
<image0>

Which option shows the same shape as the reference, just rotated?

A: <image1>
B: <image2>

Answer with one letter: A or B.

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


---

## Phase 2 — Enhanced Parsing

Same prompt as baseline, but with robust fallback parsing:
- `"Image A"`, `"Option A"`, `"Picture A"` patterns
- `"I choose/select A"`, `(A)` extraction
- Reverse sentence scan, last standalone letter A–B

**Rationale:** Binary choice (A/B) is simpler to parse, but the model may still wrap answers in phrases.

In [7]:
uid2 = "mrot_3d_shape_240_an1-240"
print("=== PROMPT (Phase 2 — same prompt, parsing differs) ===")
print(build_prompt(set(), uid2))
r_base = load_result("phase_baseline.csv", uid2)
r_p2 = load_result("phase_2.csv", uid2)
if r_base:
    print(f"\n--- Baseline parse ---")
    print(f"  Response: {r_base['raw_response']}")
    print(f"  Predicted: {r_base['predicted_label']}  Parsed: {r_base['parsed']}")
if r_p2:
    print(f"\n--- Phase 2 parse ---")
    print(f"  Response: {r_p2['raw_response']}")
    print(f"  Predicted: {r_p2['predicted_label']}  Parsed: {r_p2['parsed']}")

=== PROMPT (Phase 2 — same prompt, parsing differs) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

--- Baseline parse ---
  Response: B
  Predicted: B  Parsed: True

--- Phase 2 parse ---
  Response: B
  Predicted: B  Parsed: True


---

## Phase 3 — Task-Specific System Prompt

Replaces the generic Qwen system prompt with:
- **System:** `"You are a spatial reasoning assistant. You compare shapes to determine which option matches a rotated reference image. Always respond with exactly one letter: A or B."`
- **Suffix:** `"Respond with exactly one letter: A or B. Nothing else."`

**Rationale:** Priming for spatial/rotation reasoning may activate better visual comparison strategies.

In [8]:
uid3 = "mrot_2d_rabbit_160_Rn-160"
print("=== PROMPT (Phase 3) ===")
print(build_prompt({3}, uid3))
r = load_result("phase_3.csv", uid3)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 3) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

Respond with exactly one letter: A or B. Nothing else.

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


---

## Phase 4 — Mirror Awareness Hint

Prepends a hint about the rotation-vs-reflection distinction:

```
One option is the same shape rotated to a different angle.
The other option is its mirror image (horizontally flipped).
Look carefully at the orientation and asymmetric features to
distinguish between rotation and reflection.
```

**Rationale:** This directly tells the model what the task is testing — chirality discrimination. Without this, the model may not realize that one option is a mirror flip.

In [9]:
uid4 = "mrot_2d_rabbit_120_Rp-120"
print("=== PROMPT (Phase 4) ===")
print(build_prompt({4}, uid4))
r = load_result("phase_4.csv", uid4)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 4) ===
One option is the same shape rotated to a different angle. The other option is its mirror image (horizontally flipped). Look carefully at the orientation and asymmetric features to distinguish between rotation and reflection.

<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: A  ✗


---

## Phase 5 — Feature-Based CoT

Appends step-by-step reasoning tailored to item type:

**For 2D silhouettes (type 2):**
```
Step 1: Look at the reference silhouette — note which direction key features point (ears, beak, tail).
Step 2: For each option, check if those features point the same way (rotated) or are flipped.
Step 3: Choose the option that matches the reference's chirality.
```

**For 3D shapes (type 3):**
```
Step 1: Identify a distinctive part of the 3D shape (protruding arm or corner).
Step 2: For each option, check if that part is in the same relative position or mirrored.
Step 3: Choose the option that is a rotation, not a mirror.
```

**Rationale:** Explicit feature-tracking scaffolds the spatial comparison that humans naturally do when solving mental rotation tasks.

In [10]:
# CoT for 2D silhouette
uid5_2d = "mrot_2d_duck_200_Dp-200"
print("=== PROMPT (Phase 5, 2D) ===")
print(build_prompt({5}, uid5_2d))
print()

# CoT for 3D shape
uid5_3d = "mrot_3d_shape_320_bp15-320"
print("=== PROMPT (Phase 5, 3D) ===")
print(build_prompt({5}, uid5_3d))

r = load_result("phase_5.csv", uid5_2d)
if r:
    print(f"\n=== MODEL OUTPUT (2D) ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 5, 2D) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

Step 1: Look at the reference silhouette — note which direction key features point (e.g., ears, beak, tail).
Step 2: For each option, check if those features point the same way (just rotated) or are flipped.
Step 3: Choose the option that matches the reference's chirality.
State your final answer as a single letter.

=== PROMPT (Phase 5, 3D) ===
<image0> Choose the option that matches this image. Answer with A or B. A: <image1>; B: <image2>

Step 1: Identify a distinctive part of the 3D shape in the reference (e.g., a protruding arm or corner).
Step 2: For each option, check if that distinctive part is in the same relative position (rotated) or mirrored.
Step 3: Choose the option that is a rotation, not a mirror.
State your final answer as a single letter.

=== MODEL OUTPUT (2D) ===
Response: B
Predicted: B  Correct: B  ✓


---

## Combined Runs

### Phase 1+2+3 — Structural Improvements

Structured prompt + enhanced parsing + spatial system prompt.

In [11]:
uid_c1 = "mrot_2d_rabbit_040_Rn-040"
print("=== Phase 1+2+3 ===")
print(build_prompt({1, 2, 3}, uid_c1))
r = load_result("phase_1_2_3.csv", uid_c1)
if r:
    print(f"\nResponse: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== Phase 1+2+3 ===
Reference image:
<image0>

Which option shows the same shape as the reference, just rotated?

A: <image1>
B: <image2>

Answer with one letter: A or B.

Respond with exactly one letter: A or B. Nothing else.

Response: B
Predicted: B  Correct: A  ✗


### Phase 1+4 — Structured + Mirror Hint

Structured prompt with the chirality/mirror awareness hint.

In [12]:
uid_c2 = "mrot_3d_shape_280_ap1-280"
print("=== Phase 1+4 ===")
print(build_prompt({1, 4}, uid_c2))
r = load_result("phase_1_4.csv", uid_c2)
if r:
    print(f"\nResponse: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== Phase 1+4 ===
One option is the same shape rotated to a different angle. The other option is its mirror image (horizontally flipped). Look carefully at the orientation and asymmetric features to distinguish between rotation and reflection.

Reference image:
<image0>

Which option shows the same shape as the reference, just rotated?

A: <image1>
B: <image2>

Answer with one letter: A or B.

Response: B
Predicted: B  Correct: B  ✓


### Phase 1+2+3+4+5 — All Improvements

Full pipeline: structured + parsing + spatial system + mirror hint + feature CoT.

In [13]:
uid_all = "mrot_2d_rabbit_080_Rp-080"
print("=== Phase 1+2+3+4+5 ===")
print(build_prompt({1, 2, 3, 4, 5}, uid_all))
r = load_result("phase_1_2_3_4_5.csv", uid_all)
if r:
    print(f"\nResponse: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== Phase 1+2+3+4+5 ===
One option is the same shape rotated to a different angle. The other option is its mirror image (horizontally flipped). Look carefully at the orientation and asymmetric features to distinguish between rotation and reflection.

Reference image:
<image0>

Which option shows the same shape as the reference, just rotated?

A: <image1>
B: <image2>

Answer with one letter: A or B.

Step 1: Look at the reference silhouette — note which direction key features point (e.g., ears, beak, tail).
Step 2: For each option, check if those features point the same way (just rotated) or are flipped.
Step 3: Choose the option that matches the reference's chirality.
State your final answer as a single letter.

Respond with exactly one letter: A or B. Nothing else.

Response: B
Predicted: B  Correct: B  ✓


---

## Conclusion

**Key questions to answer after running:**

1. **Is baseline above chance (50%)?** Mental rotation is a fundamental spatial skill — does 0.8B have any spatial reasoning ability?
2. **Does the mirror hint help?** Explicitly framing the task as rotation-vs-reflection might be the single most impactful improvement.
3. **2D vs 3D performance:** 2D silhouettes have obvious asymmetric features (ears, beak). 3D shapes are more abstract and harder to distinguish.
4. **Does CoT help or hurt?** Feature-tracking is natural for humans but may not map well to how VLMs process images.
5. **Rotation angle effect:** Higher rotation angles (160°–320°) should be harder than small angles (20°–40°).

**Limitations:**
- 83 items with 50% chance — need substantial improvement to be meaningful.
- Spatial reasoning may be fundamentally limited in sub-1B VLMs.
- Results from 0.8B may not transfer to larger models with better spatial encoders.